In [28]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.width', 0)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 50)


In [29]:
path = 'dataset/cleaned/domain.csv'
path_nsw = 'dataset/cleaned/nsw.csv'

In [30]:
df = pd.read_csv(path)
df_nsw = pd.read_csv(path_nsw)

### 1. Street-Level Market Dynamics in NSW: Sales Volume and Price Statistics

In [31]:
street_stats = (
    df_nsw.groupby('Property street name', as_index=True)
          .agg(
              total_sales=('Purchase price', 'size'),
              avg_price=('Purchase price', 'mean'),
              min_price=('Purchase price', 'min'),
              max_price=('Purchase price', 'max')
          )
          .sort_values('total_sales', ascending=False)
          .head(10)
)
street_stats

,total_sales,avg_price,min_price,max_price
Property street name,,,,
nancarrow ave,15638,"809,969",518000,16612590
pacific hwy,7849,"1,128,700",2126,15900000
george st,7842,"1,155,463",752,11187000
pitt st,6743,"794,496",3100,11500000
gladstone st,6075,"723,422",20000,9600000
princes hwy,4371,"924,977",1000,10780000
church st,4088,"840,212",730,11000000
king st,3452,"985,030",1050,10325000
victoria st,3396,"1,142,211",2110,13720000


### 2. Seller-Level Market Concentration in NSW

In [32]:
sale_stats = (
    df_nsw.groupby('Sale counter', as_index=True)['Purchase price']
          .agg(['mean', 'median', 'count'])
          .reset_index()
)

total_sales = sale_stats['count'].sum()
sale_stats['percentage'] = (sale_stats['count'] / total_sales * 100).round(2)
sale_stats.head(10)

,Sale counter,mean,median,count,percentage
0,1,"781,312","550,000",46437,3
1,2,"780,461","550,000",44223,3
2,3,"810,166","565,000",41948,3
3,4,"827,969","585,000",39529,3
4,5,"861,487","610,000",37241,2
5,6,"881,017","627,000",35132,2
6,7,"905,487","649,000",33291,2
7,8,"922,382","659,000",31689,2
8,9,"950,518","676,000",30243,2
9,10,"968,999","685,000",28993,2


### 3. Housing Infrastructure and Price Correlation: Parking as a Case Study

In [33]:
df['has_parking'] = df['Parking'] > 0

pivot_price = (
    df.pivot_table(
        index='state',
        columns='has_parking',
        values='Price',
        aggfunc='mean'
    )
    .rename(columns={False: 'avg_price_no_parking', True: 'avg_price_has_parking'})
)

parking_ratio = (
    df.groupby('state', as_index=True)['has_parking']
      .mean()
      .rename('pct_with_parking')
)
parking_summary = pivot_price.join(parking_ratio)

parking_summary['pct_with_parking'] = (parking_summary['pct_with_parking'] * 100).round(1)
parking_summary = parking_summary.round(0).sort_values('pct_with_parking', ascending=False)

parking_summary

,avg_price_no_parking,avg_price_has_parking,pct_with_parking
state,,,
act,"527,483","748,061",90
nt,"382,250","443,495",86
qld,"535,087","976,212",77
nsw,"1,047,989","1,234,091",75
vic,"570,499","867,532",75
wa,"528,914","743,627",68
sa,"343,581","764,967",60
tas,"328,419","706,071",56


### 4. Dominant Property Type per Suburb

In [34]:
counts = df.groupby(['suburb', 'Type']).size().unstack(fill_value=0)

dominant_type = counts.idxmax(axis=1)               
dominant_count = counts.max(axis=1)              
total_properties = counts.sum(axis=1)          
dominant_ratio = (dominant_count / total_properties * 100).round(1)

avg_price_all = df.groupby('suburb')['Price'].mean().round(0)

grp_mean = df.groupby(['suburb', 'Type'])['Price'].mean()  # Series with MultiIndex

avg_price_dominant = pd.Series(
    [grp_mean.get((s, dominant_type.loc[s]), np.nan) for s in dominant_type.index],
    index=dominant_type.index
).round(0)

price_std = df.groupby('suburb')['Price'].std()
price_mean = df.groupby('suburb')['Price'].mean()
price_cv = (price_std / price_mean).replace([np.inf, -np.inf], np.nan).round(3)

dominant_table = pd.DataFrame({
    'Dominant_Type': dominant_type,
    'Count': dominant_count,
    'Total_Properties': total_properties,
    'Dominant_Ratio(%)': dominant_ratio,
    'Avg_Price_All': avg_price_all,
    'Avg_Price_Dominant': avg_price_dominant,
    'Price_CV': price_cv
})

dominant_table = dominant_table.sort_values('Total_Properties', ascending=False).head(10)
dominant_table


,Dominant_Type,Count,Total_Properties,Dominant_Ratio(%),Avg_Price_All,Avg_Price_Dominant,Price_CV
suburb,,,,,,,
paddington,Terrace,154,172,90,"2,979,672","3,227,237",0
surry hills,Terrace,83,130,64,"1,674,175","2,312,449",1
darlinghurst,Terrace,61,111,55,"1,586,198","2,352,935",1
port macquarie,Villa,25,110,23,"798,911","647,020",1
melbourne,Studio,54,83,65,"391,461","210,196",1
shell cove,New land,47,83,57,"1,645,940","1,800,426",0
schofields,New apartments / off the plan,48,74,65,"759,307","693,755",0
austral,New house and land,35,66,53,"1,159,803","1,143,385",1
potts point,Studio,59,66,89,"883,091","532,271",1


### 5. Housing Price Decay with Distance from Sydney

In [35]:
price_distance = (
    df.assign(dist_to_sydney = (df['dist_to_sydney'] // 5) * 5)
          .groupby('dist_to_sydney')
          .agg(
              avg_price=('Price', 'mean'),
              min_price=('Price', 'min'),
              max_price=('Price', 'max'),
              std_price=('Price', 'std'),
              count=('Price', 'size')
          )
          .assign(price_cv=lambda d: (d['std_price'] / d['avg_price']).round(3))
          .sort_index()
)
price_distance.head(10)


,avg_price,min_price,max_price,std_price,count,price_cv
dist_to_sydney,,,,,,
0,"1,719,257","117,000","7,450,000","1,296,527",1216,1
5,"1,699,731","290,000","7,100,000","1,306,020",472,1
10,"1,615,055","305,000","7,000,000","903,449",589,1
15,"1,438,891","185,000","7,100,000","702,141",681,0
20,"1,345,609","179,000","5,000,000","625,528",543,0
25,"1,042,938","282,000","2,050,000","425,644",217,0
30,"972,192","405,000","6,500,000","615,665",219,1
35,"1,025,380","305,000","7,200,000","706,393",341,1
40,"1,073,009","245,000","6,250,225","609,386",185,1
